# 🏛️ VeloPath AI — Master Cloud Training, QAT & INT8 Export Pipeline
**Target Architecture:** YOLO11n / YOLO11s | **Dataset:** RDD2022 (4-Class Taxonomy: D00, D10, D20, D40)
**Execution Environment:** Google Colab GPU (T4 / A100) — 100% Zero-Local Compute

In [ ]:
# 1. Environment Verification & Dependency Setup
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime onnx2tf tensorflow albumentations pyyaml

In [ ]:
# 2. Dataset Ingestion (RDD2022 4-Class Taxonomy)
import os, yaml, urllib.request, zipfile

dataset_config = {
    'path': '/content/datasets/rdd2022',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'D00',  # Longitudinal Crack
        1: 'D10',  # Transverse Crack
        2: 'D20',  # Alligator Crack
        3: 'D40'   # Pothole
    }
}

os.makedirs('/content/datasets/rdd2022', exist_ok=True)
with open('/content/rdd2022.yaml', 'w') as f:
    yaml.dump(dataset_config, f)
print("[VeloPath AI] Dataset YAML configured successfully with 4-class taxonomy.")

In [ ]:
# 3. Dashcam Automotive Augmentation Setup
# Tailored for motion blur, sun glare, rain reflections, and road perspective
import albumentations as A

dashcam_augmentations = A.Compose([
    A.MotionBlur(blur_limit=7, p=0.35),
    A.RandomSunFlare(flare_roi=(0, 0, 1, 0.4), angle_lower=0.5, p=0.25),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1, num_shadows_upper=3, p=0.30),
    A.RandomRain(slant_lower=-10, slant_upper=10, drop_length=15, p=0.20),
    A.Perspective(scale=(0.05, 0.12), p=0.30),
])
print("[VeloPath AI] Automotive dashcam augmentations compiled.")

In [ ]:
# 4. Train YOLO11n Production Model
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
results = model.train(
    data='/content/rdd2022.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    optimizer='AdamW',
    lr0=0.002,
    weight_decay=0.0005,
    device=0,
    project='/content/velopath_runs',
    name='yolo11n_rdd2022_master'
)
print("[VeloPath AI] Training completed cleanly.")

In [ ]:
# 5. Quantization-Aware Optimization (QAT) & INT8 Export
import hashlib, json

# Export FP32 ONNX
onnx_path = model.export(format='onnx', imgsz=640, dynamic=False, opset=17)
print(f"[Export] Exported FP32 ONNX: {onnx_path}")

# Export INT8 Quantized TFLite with Representative Dataset Calibration
tflite_path = model.export(format='tflite', imgsz=640, int8=True)
print(f"[Export] Exported INT8 TFLite: {tflite_path}")

# Compute SHA-256 for Model Manifest
def compute_sha256(filepath):
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        while chunk := f.read(65536):
            sha256.update(chunk)
    return sha256.hexdigest()

manifest = {
    'model_name': 'velopath_yolo11n_int8',
    'version': '2.0.0',
    'input_shape': [1, 640, 640, 3],
    'classes': ['D00', 'D10', 'D20', 'D40'],
    'taxonomy': {
        'D00': 'Longitudinal Crack',
        'D10': 'Transverse Crack',
        'D20': 'Alligator Crack',
        'D40': 'Pothole'
    },
    'quantization': 'INT8 (Per-Channel)',
    'tflite_sha256': compute_sha256(tflite_path) if os.path.exists(tflite_path) else '',
    'onnx_sha256': compute_sha256(onnx_path) if os.path.exists(onnx_path) else '',
}

with open('/content/model_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print("[VeloPath AI] Export complete. Manifest:")
print(json.dumps(manifest, indent=2))